# Peer comparison: develop and test without a server

Install `python -m pip install -e '.[dev]'` in this kernel's environment.
The implementation is ordinary Python: `PeerComparison.run`, not copied here.
This notebook uses strict fixtures first. It makes no live provider calls.

Run from the repository root with the package installed.


In [ ]:
from examples.methods.peer_comparison.fixtures import fixture_provider, BINDINGS, PARAMS, DATE, DATES
from decision_layer.dev import MethodSession
from decision_layer.methods.context import Scope
from decision_layer.methods.peer_comparison import PeerComparison
from decision_layer.semantic.credentials import AnonymousServiceCredentials

provider = fixture_provider()
session = await MethodSession.connect(provider, AnonymousServiceCredentials())
session.register(PeerComparison())
[(obj.title, obj.ref) for obj in session.metrics()]


## Execute in the kernel

Put a breakpoint in PeerComparison.run to inspect target, peer and overall.
Each aggregate is computed by the semantic provider, not an average of sellers.


In [ ]:
trial = await session.run(
    "query.peer_comparison", bindings=BINDINGS, params=PARAMS,
    scope=Scope(date_range=DATES, time_dimension=DATE),
)
assert trial.result.status == "success"
assert [row["difference_from_subject"] for row in trial.result.primary.data["rows"]] == [0, 4, 6]
provider.assert_consumed()
trial.result.primary.data["rows"]


## Inspect all three queries

Exact fixture specs verify the period, peer restrictions and subject exclusion.
No significance or causal effect is claimed.


In [ ]:
for attempt in trial.attempts:
    print(attempt.status, attempt.spec.model_dump(mode="json"))
print(trial.result.validation)
print(trial.result.warnings)


## Preview a Recipe through the ordinary Run engine

A new fixture provider supplies new responses; fixtures are consumed once.
Preview runs in workers and stays in memory. It is not uploaded to a Web server.


In [ ]:
from decision_layer.core.models import PlanStep, Recipe, SemanticScope

preview_session = await MethodSession.connect(fixture_provider(), AnonymousServiceCredentials())
preview_session.register(PeerComparison())
recipe = Recipe(
    name="peer-example", version="1.0.0", mode="pipeline",
    description="Compare a selected subject to explicit peers",
    semantic_scope=SemanticScope(primary_metric=BINDINGS["metric"]),
    steps=[PlanStep(id="compare", method="query.peer_comparison", bindings=BINDINGS, params=PARAMS)],
)
run = await preview_session.preview(
    recipe, step_index=0, scope={"date_range": list(DATES), "time_dimension": DATE},
)
assert run.steps[0].result.status == "success"
assert len(run.query_attempts) == 3
print(run.id, run.status)
preview_session.close()


## Use a real source after deterministic tests

Use MethodSession.cube(url, RequestCredentials(token)), discover session.metrics()
and session.dimensions(metric_ref), and select real refs explicitly. Do not reuse
fixture references or store tokens in cells/outputs. examples/methods/first_method/explore.ipynb shows the
real-source path. Run pytest examples/methods/peer_comparison to test failures too.
